# Détection Multi-Appareils — House 3 (Test)

Ce notebook applique les **4 modèles règle-based** (meilleure performance) entraînés
dans les notebooks individuels (`kettle1`, `fridge1`, `microwave1`, `tv1`) pour détecter
l'état de 4 appareils dans **House 3** à partir du seul signal de puissance agrégée.

### Appareils cibles dans House 3
| Index (0-based) | Appareil |
|---|---|
| 0 | Aggregate |
| 1 | Toaster |
| **2** | **Fridge-Freezer** ← détecté |
| 3 | Freezer |
| 4 | Tumble Dryer |
| 5 | Dishwasher |
| 6 | Washing Machine |
| **7** | **Television** ← détecté |
| **8** | **Microwave** ← détecté |
| **9** | **Kettle** ← détecté |

### Approche
- Chaque modèle apprend la puissance ON à partir de maisons d'entraînement (House 3 exclue du train)
- L'algorithme **règle événementielle** (meilleure performance) est appliqué sur l'agrégé de House 3
- Évaluation : Précision / Rappel / F1-Score + MAE (W)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    mean_absolute_error,
)

# ── Global config ─────────────────────────────────────────────────────
DATA_DIR   = "data"
BASE_FREQ  = "10s"           # Ré-échantillonnage à 10 secondes

TEST_HOUSE = 3               # House 3 est notre maison de test unique

# House 3 – mapping des colonnes (index 0-based des sous-compteurs)
# House 3: 0=Aggregate, 1=Toaster, 2=Fridge-Freezer, 3=Freezer,
#          4=Tumble Dryer, 5=Dishwasher, 6=Washing Machine,
#          7=Television, 8=Microwave, 9=Kettle
HOUSE3_APPLIANCE_COLS = {
    "Kettle"        : 9,
    "Fridge-Freezer": 2,
    "Microwave"     : 8,
    "Television"    : 7,
}

# Fenêtre (en secondes) avant un événement utilisée pour estimer la baseline
PRE_EVENT_WINDOW_SECONDS = 60

# ── Appliance-specific mappings (for training houses) ─────────────────
# Chaque dictionnaire : {house_id: col_index_0based_dans_le_CSV}
# Kettle : House 3 était dans le train original → retiré ici pour un test loyal
kettle_map = {
    2: 8, 3: 9, 4: 9, 5: 8, 6: 7, 7: 9, 8: 9, 9: 7, 12: 4, 13: 9, 19: 5, 20: 9
}
kettle_train_houses = [4, 6, 7, 8, 9, 12, 13, 19, 20]   # house 3 excluded

# Fridge : house 3 col 2 ajouté ; house 3 PAS dans le train original → inchangé
fridge_map = {2: 1, 3: 2, 5: 1, 9: 1, 12: 1, 15: 1}
fridge_train_houses = [2, 5, 9]                          # unchanged

# Microwave : house 3 col 8 ajouté ; house 3 PAS dans le train original → inchangé
microwave_map = {3: 8, 4: 8, 10: 8, 12: 3, 17: 7, 19: 4}
microwave_train_houses = [10, 12, 19]                    # unchanged

# Television : House 3 était dans le train original → retiré ici pour un test loyal
television_map = {
    1: 8, 2: 4, 3: 7, 4: 7, 5: 6, 6: 5, 7: 7, 8: 7, 9: 5, 10: 7,
    12: 2, 13: 1, 15: 6, 16: 8, 17: 6, 18: 8, 19: 3, 20: 7, 21: 6,
}
television_train_houses = [1, 2, 4, 5, 6, 7, 8, 9, 10]  # house 3 excluded

print("Configuration chargée.")
print(f"  Maison de test  : House {TEST_HOUSE}")
print(f"  Appareils cibles: {list(HOUSE3_APPLIANCE_COLS.keys())}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# IO + Prétraitement
# ─────────────────────────────────────────────────────────────────────

def load_refit_house(house_id: int, appliance_col_idx_0based: int,
                     appliance_name: str = "Appliance") -> pd.DataFrame:
    """
    Charge Aggregate + un sous-compteur d'appareil pour une maison REFIT.
    Returns a DataFrame with columns ['Aggregate', appliance_name].
    """
    file_path = f"{DATA_DIR}/House_{house_id}.csv"
    df = pd.read_csv(
        file_path,
        usecols=[1, 2, appliance_col_idx_0based + 2],
        header=0,
        names=["Unix", "Aggregate", appliance_name],
    )
    df["Timestamp"] = pd.to_datetime(df["Unix"], unit="s")
    df = df.set_index("Timestamp").sort_index()
    df = df[["Aggregate", appliance_name]].astype(float)
    full_range = pd.date_range(df.index.min(), df.index.max(), freq=BASE_FREQ)
    df = df.reindex(full_range).ffill().fillna(0.0)
    return df


def load_refit_aggregate_only(house_id: int) -> np.ndarray:
    """Charge uniquement le signal agrégé (pour l'entraînement HMM rapide)."""
    fp = f"{DATA_DIR}/House_{house_id}.csv"
    return (
        pd.read_csv(fp, usecols=[2], header=0, names=["Aggregate"])
        ["Aggregate"].astype(float).values
    )


def remove_outliers(series: np.ndarray, k: float = 4.0) -> np.ndarray:
    """Remplace les valeurs aberrantes par interpolation (filtre k-sigma robuste)."""
    med = np.median(series)
    mad = np.median(np.abs(series - med)) * 1.4826
    threshold = k * mad
    cleaned = series.copy()
    mask = np.abs(series - med) > threshold
    for i in np.where(mask)[0]:
        cleaned[i] = cleaned[i - 1] if i > 0 else med
    return cleaned


def median_filter(series: np.ndarray, window: int = 3) -> np.ndarray:
    """Filtre médian glissant anti-bruit."""
    half = window // 2
    n = len(series)
    result = np.empty(n)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        result[i] = np.median(series[lo:hi])
    return result


def preprocess_aggregate(aggregate: np.ndarray,
                          apply_outlier_removal: bool = True,
                          apply_median_filter: bool = True,
                          median_window: int = 3) -> np.ndarray:
    """Pipeline de prétraitement : outlier removal + filtre médian."""
    sig = aggregate.astype(float)
    if apply_outlier_removal:
        sig = remove_outliers(sig)
    if apply_median_filter:
        sig = median_filter(sig, window=median_window)
    return sig


print("Fonctions IO et prétraitement définies.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Algorithmes de détection règle-based (un par appareil)
# ─────────────────────────────────────────────────────────────────────

SAMPLE_SECONDS = int(pd.to_timedelta(BASE_FREQ).total_seconds())


# ── 1. Détection Bouilloire (Kettle) ─────────────────────────────────
KETTLE_MIN_ON   = 90           # s
KETTLE_MAX_ON   = 15 * 60      # s
KETTLE_STEP_ON  = 0.75
KETTLE_STEP_OFF = 0.65
KETTLE_TOL      = 0.15
KETTLE_STANDBY  = 200.0        # W


def learn_kettle_on_power(train_houses, kettle_map) -> float:
    samples = []
    for h in train_houses:
        df = load_refit_house(h, kettle_map[h], "Kettle")
        k = df["Kettle"].values
        on = k[k > KETTLE_STANDBY]
        if len(on) > 0:
            samples.append(on)
    if not samples:
        raise RuntimeError("No kettle ON samples found.")
    return float(np.quantile(np.concatenate(samples), 0.70))


def detect_kettle(aggregate: np.ndarray, kettle_on_power: float) -> np.ndarray:
    """Détection règle-based : montée/descente de step + validation plateau."""
    n = len(aggregate)
    pred = np.zeros(n, dtype=float)
    d = np.diff(aggregate, prepend=aggregate[0])
    step_on  = KETTLE_STEP_ON  * kettle_on_power
    step_off = KETTLE_STEP_OFF * kettle_on_power
    tol      = KETTLE_TOL      * kettle_on_power
    i = 1
    while i < n:
        if d[i] >= step_on:
            start = i
            j, end = i + 1, None
            while j < n and (j - start) * SAMPLE_SECONDS <= KETTLE_MAX_ON:
                if d[j] <= -step_off:
                    end = j
                    break
                j += 1
            if end is None:
                i += 1
                continue
            dur = (end - start) * SAMPLE_SECONDS
            if dur < KETTLE_MIN_ON:
                i = end + 1
                continue
            pre_w  = aggregate[max(0, start - int(PRE_EVENT_WINDOW_SECONDS / SAMPLE_SECONDS)):start]
            on_w   = aggregate[start:end]
            if len(pre_w) < 3 or len(on_w) < 3:
                i = end + 1
                continue
            delta = float(np.median(on_w)) - float(np.median(pre_w))
            if abs(delta - kettle_on_power) <= tol:
                pred[start:end] = kettle_on_power
            i = end + 1
        else:
            i += 1
    return pred


# ── 2. Détection Réfrigérateur (Fridge-Freezer) ──────────────────────
FRIDGE_MIN_ON   = 60           # s
FRIDGE_MAX_ON   = 60 * 90      # s
FRIDGE_STEP_SMOOTH = 15        # s
FRIDGE_STEP_ON  = 0.12
FRIDGE_STEP_OFF = 0.12
FRIDGE_TOL      = 0.75
FRIDGE_STANDBY  = 10.0         # W


def learn_fridge_on_power(train_houses, fridge_map) -> float:
    samples = []
    for h in train_houses:
        df = load_refit_house(h, fridge_map[h], "Fridge")
        f = df["Fridge"].values
        on = f[f > FRIDGE_STANDBY]
        if len(on) > 0:
            samples.append(on)
    if not samples:
        raise RuntimeError("No fridge ON samples found.")
    return float(np.quantile(np.concatenate(samples), 0.70))


def detect_fridge(aggregate: np.ndarray, fridge_on_power: float) -> np.ndarray:
    """Détection règle-based (step-based + résiduel) pour réfrigérateur."""
    n = len(aggregate)
    step_win       = max(3, int(FRIDGE_STEP_SMOOTH / SAMPLE_SECONDS))
    min_on_samp    = max(1, int(FRIDGE_MIN_ON / SAMPLE_SECONDS))
    max_on_samp    = int(FRIDGE_MAX_ON / SAMPLE_SECONDS)
    step_on_th     = FRIDGE_STEP_ON  * fridge_on_power
    step_off_th    = FRIDGE_STEP_OFF * fridge_on_power
    tol            = FRIDGE_TOL      * fridge_on_power

    # Smoothed step signal
    cum = np.cumsum(np.concatenate([[0.0], aggregate.astype(float)]))
    k_idx = np.arange(step_win, n - step_win)
    pre_mean  = (cum[k_idx]            - cum[k_idx - step_win]) / step_win
    post_mean = (cum[k_idx + step_win] - cum[k_idx])            / step_win
    smooth_step = np.zeros(n)
    smooth_step[step_win:n - step_win] = post_mean - pre_mean

    pred_primary = np.zeros(n, dtype=float)
    i = step_win
    while i < n - step_win:
        if smooth_step[i] >= step_on_th:
            start = i
            j, end = start + min_on_samp, None
            while j < min(n - step_win, start + max_on_samp):
                if smooth_step[j] <= -step_off_th:
                    end = j
                    break
                j += 1
            if end is None:
                end = min(n - step_win, start + max_on_samp)
            pre_w = aggregate[max(0, start - step_win):start]
            on_w  = aggregate[start:end]
            if len(pre_w) >= 2 and len(on_w) >= 2:
                delta = float(np.median(on_w)) - float(np.median(pre_w))
                if abs(delta - fridge_on_power) <= tol:
                    pred_primary[start:end] = fridge_on_power
            i = end + 1
        else:
            i += 1

    # Residual-based secondary detector
    baseline_win = max(step_win, int(30 * 60 / SAMPLE_SECONDS))
    baseline = (
        pd.Series(aggregate)
        .rolling(window=baseline_win, min_periods=1, center=False)
        .quantile(0.15)
        .values
    )
    residual  = aggregate.astype(float) - baseline
    candidate = np.abs(residual - fridge_on_power) <= tol

    pred_secondary = np.zeros(n, dtype=float)
    in_event, start_ev = False, 0
    for k in range(n):
        if candidate[k] and not in_event:
            in_event, start_ev = True, k
        elif not candidate[k] and in_event:
            if k - start_ev >= min_on_samp:
                pred_secondary[start_ev:k] = fridge_on_power
            in_event = False
    if in_event and n - start_ev >= min_on_samp:
        pred_secondary[start_ev:n] = fridge_on_power

    return np.maximum(pred_primary, pred_secondary)


# ── 3. Détection Micro-ondes (Microwave) ─────────────────────────────
MICRO_MIN_ON   = 20            # s
MICRO_MAX_ON   = 20 * 60       # s
MICRO_STEP_ON  = 0.60
MICRO_STEP_OFF = 0.50
MICRO_TOL      = 0.25
MICRO_STANDBY  = 50.0          # W


def learn_microwave_on_power(train_houses, microwave_map) -> float:
    samples = []
    for h in train_houses:
        df = load_refit_house(h, microwave_map[h], "Microwave")
        m = df["Microwave"].values
        on = m[m > MICRO_STANDBY]
        if len(on) > 0:
            samples.append(on)
    if not samples:
        raise RuntimeError("No microwave ON samples found.")
    return float(np.quantile(np.concatenate(samples), 0.70))


def detect_microwave(aggregate: np.ndarray, microwave_on_power: float) -> np.ndarray:
    """Détection règle-based : montée/descente de step + validation plateau."""
    n = len(aggregate)
    pred = np.zeros(n, dtype=float)
    d = np.diff(aggregate, prepend=aggregate[0])
    step_on  = MICRO_STEP_ON  * microwave_on_power
    step_off = MICRO_STEP_OFF * microwave_on_power
    tol      = MICRO_TOL      * microwave_on_power
    i = 1
    while i < n:
        if d[i] >= step_on:
            start = i
            j, end = i + 1, None
            while j < n and (j - start) * SAMPLE_SECONDS <= MICRO_MAX_ON:
                if d[j] <= -step_off:
                    end = j
                    break
                j += 1
            if end is None:
                i += 1
                continue
            dur = (end - start) * SAMPLE_SECONDS
            if dur < MICRO_MIN_ON:
                i = end + 1
                continue
            pre_w = aggregate[max(0, start - int(PRE_EVENT_WINDOW_SECONDS / SAMPLE_SECONDS)):start]
            on_w  = aggregate[start:end]
            if len(pre_w) < 3 or len(on_w) < 3:
                i = end + 1
                continue
            delta = float(np.median(on_w)) - float(np.median(pre_w))
            if abs(delta - microwave_on_power) <= tol:
                pred[start:end] = microwave_on_power
            i = end + 1
        else:
            i += 1
    return pred


# ── 4. Détection Télévision (Television) ─────────────────────────────
TV_MIN_ON   = 5 * 60           # s
TV_MAX_ON   = 8 * 60 * 60      # s
TV_STEP_ON  = 0.20
TV_STEP_OFF = 0.20
TV_TOL      = 0.35
TV_STANDBY  = 50.0             # W


def learn_tv_on_power(train_houses, television_map) -> float:
    samples = []
    for h in train_houses:
        df = load_refit_house(h, television_map[h], "Television")
        t = df["Television"].values
        on = t[t > TV_STANDBY]
        if len(on) > 0:
            samples.append(on)
    if not samples:
        raise RuntimeError("No TV ON samples found.")
    return float(np.quantile(np.concatenate(samples), 0.70))


def detect_tv(aggregate: np.ndarray, tv_on_power: float) -> np.ndarray:
    """Détection règle-based : montée/descente de step + validation plateau."""
    n = len(aggregate)
    pred = np.zeros(n, dtype=float)
    d = np.diff(aggregate, prepend=aggregate[0])
    step_on  = TV_STEP_ON  * tv_on_power
    step_off = TV_STEP_OFF * tv_on_power
    tol      = TV_TOL      * tv_on_power
    i = 1
    while i < n:
        if d[i] >= step_on:
            start = i
            j, end = i + 1, None
            while j < n and (j - start) * SAMPLE_SECONDS <= TV_MAX_ON:
                if d[j] <= -step_off:
                    end = j
                    break
                j += 1
            if end is None:
                i += 1
                continue
            dur = (end - start) * SAMPLE_SECONDS
            if dur < TV_MIN_ON:
                i = end + 1
                continue
            pre_w = aggregate[max(0, start - int(PRE_EVENT_WINDOW_SECONDS / SAMPLE_SECONDS)):start]
            on_w  = aggregate[start:end]
            if len(pre_w) < 3 or len(on_w) < 3:
                i = end + 1
                continue
            delta = float(np.median(on_w)) - float(np.median(pre_w))
            if abs(delta - tv_on_power) <= tol:
                pred[start:end] = tv_on_power
            i = end + 1
        else:
            i += 1
    return pred


# ── Metrics helper ────────────────────────────────────────────────────
def on_off_metrics(y_true_w, y_pred_w, on_threshold=10.0):
    y_true = (np.asarray(y_true_w) >= on_threshold).astype(int)
    y_pred = (np.asarray(y_pred_w) >= on_threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    mae = mean_absolute_error(y_true_w, y_pred_w)
    return float(p), float(r), float(f1), float(mae)


print("Toutes les fonctions de détection définies :")
print("  detect_kettle()     — règle événementielle bouilloire")
print("  detect_fridge()     — règle événementielle + résiduel réfrigérateur")
print("  detect_microwave()  — règle événementielle micro-ondes")
print("  detect_tv()         — règle événementielle télévision")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Apprentissage des puissances ON à partir des maisons d'entraînement
# ─────────────────────────────────────────────────────────────────────

print("Apprentissage des puissances ON (sous-compteurs maisons train)...")
print()

kettle_on_power   = learn_kettle_on_power(kettle_train_houses, kettle_map)
fridge_on_power   = learn_fridge_on_power(fridge_train_houses, fridge_map)
micro_on_power    = learn_microwave_on_power(microwave_train_houses, microwave_map)
tv_on_power       = learn_tv_on_power(television_train_houses, television_map)

learned = {
    "Kettle"        : (kettle_on_power,  kettle_train_houses),
    "Fridge-Freezer": (fridge_on_power,  fridge_train_houses),
    "Microwave"     : (micro_on_power,   microwave_train_houses),
    "Television"    : (tv_on_power,      television_train_houses),
}

print(f"{'Appareil':<18}  {'Puissance ON (W)':>16}  {'Maisons train'}")
print("-" * 65)
for name, (power, houses) in learned.items():
    print(f"{name:<18}  {power:>16.1f}  {houses}")

# ── Visualisation ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
names  = list(learned.keys())
powers = [learned[n][0] for n in names]
colors = ['#FF6B35', '#2196F3', '#FF9800', '#4CAF50']

bars = ax.bar(names, powers, color=colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, powers):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{val:.0f} W', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Puissance ON apprise (W)')
ax.set_title('Puissance ON apprise par appareil (70e percentile sous-compteurs train)',
             fontsize=12, fontweight='bold')
ax.yaxis.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

print("\nPuissances ON apprises avec succès.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Application des modèles sur House 3 (test)
# ─────────────────────────────────────────────────────────────────────

print(f"Chargement de House {TEST_HOUSE}...")

# Charge chaque sous-compteur
df_kettle = load_refit_house(TEST_HOUSE, HOUSE3_APPLIANCE_COLS["Kettle"],         "Kettle")
df_fridge = load_refit_house(TEST_HOUSE, HOUSE3_APPLIANCE_COLS["Fridge-Freezer"], "Fridge")
df_micro  = load_refit_house(TEST_HOUSE, HOUSE3_APPLIANCE_COLS["Microwave"],      "Microwave")
df_tv     = load_refit_house(TEST_HOUSE, HOUSE3_APPLIANCE_COLS["Television"],     "Television")

# Aligner tous les indices temporels sur l'agrégé commun
common_index = df_kettle.index
df_fridge = df_fridge.reindex(common_index).ffill().fillna(0.0)
df_micro  = df_micro.reindex(common_index).ffill().fillna(0.0)
df_tv     = df_tv.reindex(common_index).ffill().fillna(0.0)

aggregate = preprocess_aggregate(df_kettle["Aggregate"].values)

print(f"  Durée totale : {len(aggregate) * SAMPLE_SECONDS / 3600:.1f} heures ({len(aggregate):,} échantillons)")
print()
print("Détection règle-based en cours...")

# Détection
pred_kettle = detect_kettle(aggregate, kettle_on_power)
pred_fridge = detect_fridge(aggregate, fridge_on_power)
pred_micro  = detect_microwave(aggregate, micro_on_power)
pred_tv     = detect_tv(aggregate, tv_on_power)

# Stocker dans un DataFrame résultats
results = pd.DataFrame(index=common_index)
results["Aggregate"]          = aggregate
results["Kettle_True"]        = df_kettle["Kettle"].values
results["Kettle_Pred"]        = pred_kettle
results["Fridge_True"]        = df_fridge["Fridge"].values
results["Fridge_Pred"]        = pred_fridge
results["Microwave_True"]     = df_micro["Microwave"].values
results["Microwave_Pred"]     = pred_micro
results["Television_True"]    = df_tv["Television"].values
results["Television_Pred"]    = pred_tv

print("Détection terminée.")
print(f"  DataFrame résultats : {results.shape}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Métriques de performance — House 3
# ─────────────────────────────────────────────────────────────────────

appliance_cfg = [
    ("Kettle",         "Kettle_True",     "Kettle_Pred",     KETTLE_STANDBY),
    ("Fridge-Freezer", "Fridge_True",     "Fridge_Pred",     FRIDGE_STANDBY),
    ("Microwave",      "Microwave_True",  "Microwave_Pred",  MICRO_STANDBY),
    ("Television",     "Television_True", "Television_Pred", TV_STANDBY),
]

metrics = {}
print(f"{'Appareil':<18}  {'Précision':>9}  {'Rappel':>8}  {'F1-Score':>9}  {'MAE (W)':>9}")
print("-" * 65)
for name, true_col, pred_col, th in appliance_cfg:
    p, r, f1, mae = on_off_metrics(
        results[true_col].values,
        results[pred_col].values,
        on_threshold=th
    )
    metrics[name] = dict(precision=p, recall=r, f1=f1, mae=mae)
    print(f"{name:<18}  {p:>9.3f}  {r:>8.3f}  {f1:>9.3f}  {mae:>9.1f}")

# ── Graphique en barres des métriques ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

app_names  = list(metrics.keys())
colors_app = ['#FF6B35', '#2196F3', '#FF9800', '#4CAF50']

x = np.arange(len(app_names))
w = 0.25

ax = axes[0]
ax.bar(x - w, [metrics[n]['precision'] for n in app_names], w, label='Précision', color='#29B6F6', alpha=0.9, edgecolor='white')
ax.bar(x,     [metrics[n]['recall']    for n in app_names], w, label='Rappel',    color='#66BB6A', alpha=0.9, edgecolor='white')
ax.bar(x + w, [metrics[n]['f1']        for n in app_names], w, label='F1-Score',  color='#FFA726', alpha=0.9, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(app_names, rotation=15, ha='right')
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title(f'Métriques de détection ON/OFF — House {TEST_HOUSE}', fontweight='bold')
ax.legend(); ax.yaxis.grid(True, alpha=0.35)
for xi, name in zip(x, app_names):
    ax.text(xi + w, metrics[name]['f1'] + 0.02, f"{metrics[name]['f1']:.2f}",
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='#E65100')

ax2 = axes[1]
mae_vals = [metrics[n]['mae'] for n in app_names]
bars2 = ax2.bar(app_names, mae_vals, color=colors_app, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars2, mae_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.1f} W', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('MAE (W)'); ax2.set_title(f'Erreur absolue moyenne (MAE) — House {TEST_HOUSE}', fontweight='bold')
ax2.yaxis.grid(True, alpha=0.35)

plt.suptitle(f'Performance des 4 détecteurs règle-based — House {TEST_HOUSE}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Matrices de confusion — House 3
# ─────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, true_col, pred_col, th) in zip(axes, appliance_cfg):
    y_true = (results[true_col].values >= th).astype(int)
    y_pred = (results[pred_col].values >= th).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['OFF', 'ON'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['OFF', 'ON'])
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.set_title(f'{name}\nF1={metrics[name]["f1"]:.3f}', fontweight='bold')
    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black', fontweight='bold')

plt.suptitle(f'Matrices de confusion ON/OFF — House {TEST_HOUSE}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Visualisation du signal — fenêtre temporelle (3 000 échantillons)
# ─────────────────────────────────────────────────────────────────────

WIN_START = 1000
WIN_END   = WIN_START + 3000

fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)

plot_cfg = [
    ("Kettle",         "Kettle_True",     "Kettle_Pred",     '#FF6B35', KETTLE_STANDBY),
    ("Fridge-Freezer", "Fridge_True",     "Fridge_Pred",     '#2196F3', FRIDGE_STANDBY),
    ("Microwave",      "Microwave_True",  "Microwave_Pred",  '#FF9800', MICRO_STANDBY),
    ("Television",     "Television_True", "Television_Pred", '#4CAF50', TV_STANDBY),
]

for ax, (name, true_col, pred_col, color, th) in zip(axes, plot_cfg):
    agg_win  = results["Aggregate"].iloc[WIN_START:WIN_END].values
    true_win = results[true_col].iloc[WIN_START:WIN_END].values
    pred_win = results[pred_col].iloc[WIN_START:WIN_END].values
    t = np.arange(len(agg_win))

    ax.fill_between(t, 0, agg_win, color='gray', alpha=0.10, label='Agrégé')
    ax.plot(t, agg_win,  color='gray',  alpha=0.35, linewidth=0.7)
    ax.plot(t, true_win, color=color,   alpha=0.85, linewidth=1.5, label=f'{name} réel')
    ax.plot(t, pred_win, color='black', alpha=0.85, linewidth=1.5,
            linestyle='--', label='Prédit (règles)')

    f1_str = f"F1={metrics[name]['f1']:.3f}"
    ax.set_title(f'{name}  ({f1_str})', fontsize=10, fontweight='bold')
    ax.set_ylabel('Puissance (W)')
    ax.legend(fontsize=8, loc='upper right')
    ax.yaxis.grid(True, alpha=0.3)

axes[-1].set_xlabel(f'Échantillons (×{SAMPLE_SECONDS} s)  |  fenêtre : {WIN_START}–{WIN_END}')
plt.suptitle(f'Signal réel vs prédit — House {TEST_HOUSE} (règle-based)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Grille d'événements détectés (premiers événements par appareil)
# ─────────────────────────────────────────────────────────────────────

def get_events(signal, min_gap=5):
    """Extrait les segments ON (blocs continus > 0) depuis un signal."""
    events = []
    in_event, start = False, 0
    for i, v in enumerate(signal):
        if v > 0 and not in_event:
            in_event, start = True, i
        elif v == 0 and in_event:
            if i - start >= min_gap:
                events.append((start, i))
            in_event = False
    if in_event and len(signal) - start >= min_gap:
        events.append((start, len(signal)))
    return events


N_EVENTS_SHOW = 3   # événements par appareil
PAD = 60            # échantillons de contexte

fig, axes = plt.subplots(4, N_EVENTS_SHOW, figsize=(18, 14))
colors_map = {
    "Kettle"        : '#FF6B35',
    "Fridge-Freezer": '#2196F3',
    "Microwave"     : '#FF9800',
    "Television"    : '#4CAF50',
}

for row_idx, (name, true_col, pred_col, color, th) in enumerate(plot_cfg):
    pred_sig = results[pred_col].values
    true_sig = results[true_col].values
    agg_sig  = results["Aggregate"].values
    events   = get_events(pred_sig)

    for col_idx in range(N_EVENTS_SHOW):
        ax = axes[row_idx, col_idx]
        if col_idx >= len(events):
            ax.set_visible(False)
            continue
        ev_s, ev_e = events[col_idx]
        s = max(0, ev_s - PAD)
        e = min(len(pred_sig), ev_e + PAD)
        t = np.arange(e - s)

        ax.fill_between(t, 0, agg_sig[s:e], color='gray', alpha=0.10)
        ax.plot(t, agg_sig[s:e],  color='gray',  alpha=0.35, linewidth=0.7)
        ax.plot(t, true_sig[s:e], color=color,   linewidth=1.5, label='Réel')
        ax.plot(t, pred_sig[s:e], color='black', linewidth=1.5,
                linestyle='--', label='Prédit')
        ev_s_rel = ev_s - s
        ev_e_rel = ev_e - s
        ax.axvspan(ev_s_rel, ev_e_rel, alpha=0.15, color=color)
        dur = (ev_e - ev_s) * SAMPLE_SECONDS
        ax.set_title(f'{name}\nÉvt {col_idx+1} · {dur}s', fontsize=8, fontweight='bold')
        ax.set_xlabel('Échantillons', fontsize=7)
        ax.yaxis.grid(True, alpha=0.3)
        if col_idx == 0:
            ax.set_ylabel('W', fontsize=8)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=7)

plt.suptitle(f'Premiers événements détectés par appareil — House {TEST_HOUSE}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Vue globale : timeline complète des activations
# ─────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(5, 1, figsize=(20, 14), sharex=True)

# Agrégé
axes[0].plot(results["Aggregate"].values, color='gray', alpha=0.6,
             linewidth=0.5, label='Puissance agrégée')
axes[0].set_ylabel('W'); axes[0].legend(fontsize=8)
axes[0].set_title(f'Signal agrégé — House {TEST_HOUSE}', fontweight='bold')
axes[0].yaxis.grid(True, alpha=0.3)

for ax, (name, true_col, pred_col, color, th) in zip(axes[1:], plot_cfg):
    pred_sig = results[pred_col].values
    true_sig = results[true_col].values

    pred_events = get_events(pred_sig)
    true_events = get_events(true_sig)

    ax.plot(true_sig, color=color, alpha=0.6, linewidth=0.5, label=f'{name} réel')
    for i, (s, e) in enumerate(pred_events):
        ax.axvspan(s, e, alpha=0.35, color='red',
                   label='Prédit' if i == 0 else "")
    for i, (s, e) in enumerate(true_events):
        ax.axvspan(s, e, alpha=0.15, color=color,
                   label='Ground truth' if i == 0 else "")

    ax.set_ylabel('W'); ax.legend(fontsize=7, loc='upper right')
    f1_str = f"F1={metrics[name]['f1']:.3f}"
    ax.text(0.01, 0.88,
            f'{len(pred_events)} détectés · {len(true_events)} réels · {f1_str}',
            transform=ax.transAxes, fontsize=8, color='darkred', fontweight='bold')
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)

axes[-1].set_xlabel('Échantillons (×10 s)')
plt.suptitle(f'Timeline globale — House {TEST_HOUSE} : activations prédites vs réelles',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Tableau récapitulatif final
# ─────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

col_headers = ['Appareil', 'Colonne H3', 'Train (maisons)',
               'Puissance ON (W)', 'Précision', 'Rappel', 'F1-Score', 'MAE (W)']

row_data = [
    ('Kettle',         '9', str(kettle_train_houses),
     f'{kettle_on_power:.0f}',
     f'{metrics["Kettle"]["precision"]:.3f}',
     f'{metrics["Kettle"]["recall"]:.3f}',
     f'{metrics["Kettle"]["f1"]:.3f}',
     f'{metrics["Kettle"]["mae"]:.1f}'),
    ('Fridge-Freezer', '2', str(fridge_train_houses),
     f'{fridge_on_power:.0f}',
     f'{metrics["Fridge-Freezer"]["precision"]:.3f}',
     f'{metrics["Fridge-Freezer"]["recall"]:.3f}',
     f'{metrics["Fridge-Freezer"]["f1"]:.3f}',
     f'{metrics["Fridge-Freezer"]["mae"]:.1f}'),
    ('Microwave',      '8', str(microwave_train_houses),
     f'{micro_on_power:.0f}',
     f'{metrics["Microwave"]["precision"]:.3f}',
     f'{metrics["Microwave"]["recall"]:.3f}',
     f'{metrics["Microwave"]["f1"]:.3f}',
     f'{metrics["Microwave"]["mae"]:.1f}'),
    ('Television',     '7', str(television_train_houses),
     f'{tv_on_power:.0f}',
     f'{metrics["Television"]["precision"]:.3f}',
     f'{metrics["Television"]["recall"]:.3f}',
     f'{metrics["Television"]["f1"]:.3f}',
     f'{metrics["Television"]["mae"]:.1f}'),
]

tbl = ax.table(
    cellText=row_data,
    colLabels=col_headers,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 1:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FAFAFA')

plt.title(f'Tableau récapitulatif — House {TEST_HOUSE} (approche règle-based)',
          fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

print("\n" + "="*65)
print(f"  RÉSUMÉ — Détection multi-appareils sur House {TEST_HOUSE}")
print("="*65)
f1_vals = [metrics[n]['f1'] for n in metrics]
print(f"  F1 moyen (4 appareils) : {np.mean(f1_vals):.3f}")
for name in metrics:
    print(f"  {name:<18} F1 = {metrics[name]['f1']:.3f}  "
          f"Précision = {metrics[name]['precision']:.3f}  "
          f"Rappel = {metrics[name]['recall']:.3f}")
print("="*65)
